# ENTSO-E — generation mix ingestion

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
import time
import argparse
import psycopg2
import pandas as pd
from entsoe import EntsoePandasClient
from datetime import datetime, timezone, timedelta

In [6]:
# ── Config ────────────────────────────────────────────────────────
load_dotenv(find_dotenv(), override=True)

ENTSOE_TOKEN = os.environ["ENTSOE_TOKEN"]
DB_URL       = os.environ["DATABASE_URL"]

conn = psycopg2.connect(DB_URL)

def query(sql, params=None):
    return pd.read_sql(sql, conn, params=params)
    
print("DB connected ✓")

# ENTSO-E bidding zone codes
COUNTRIES = {
    "DE_LU": "10Y1001A1001A82H",
    "FR":    "10YFR-RTE------C",
    "ES":    "10YES-REE------0",
    "IT":    "10YIT-GRTN-----B",
    "GB":    "10Y1001A1001A59C",
}

# Map entsoe-py column names → our source_type values
SOURCE_MAP = {
    # ── Renewables ────────────────────────────────────────────────
    "Biomass":                          "biomass",
    "Geothermal":                       "geothermal",
    "Hydro Pumped Storage":             "hydro",
    "Hydro Run-of-river and poundage":  "hydro",
    "Hydro Water Reservoir":            "hydro",
    "Marine":                           "other",
    "Other renewable":                  "other",
    "Solar":                            "solar_pv",
    "Waste":                            "other",
    "Wind Offshore":                    "wind_offshore",
    "Wind Onshore":                     "wind_onshore",
    "Other":                            "other",

    # ── Fossil ────────────────────────────────────────────────────
    "Fossil Gas":                       "gas",
    "Fossil Coal-derived gas":          "gas",
    "Fossil Hard coal":                 "coal",
    "Fossil Brown coal/Lignite":        "coal",
    "Fossil Oil":                       "oil",
    "Fossil Oil shale":                 "oil",       
    "Fossil Peat":                      "coal",      

    # ── Nuclear ───────────────────────────────────────────────────
    "Nuclear":                          "nuclear",

    # ── Other ─────────────────────────────────────────────────────
    "Energy storage":                   "other",     # similar to pumped hydro, neg values possible
                                                     
}

DB connected ✓


In [3]:
# ── DB helpers ────────────────────────────────────────────────────

def log_note(conn, country, note_type, description):
    with conn.cursor() as cur:
        cur.execute(
            "INSERT INTO data_notes (country, note_type, description) VALUES (%s, %s, %s)",
            (country, note_type, description),
        )
    conn.commit()


def insert_generation(conn, country: str, df: pd.DataFrame):

    # Flatten MultiIndex if present
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [c[0] for c in df.columns]

    # Resample 15-min data to hourly — ENTSO-E returns 15-min for most countries
    df = df.resample('h').mean()

    rows = []
    for ts, row in df.iterrows():
        # Convert timestamp to a standardized UTC ISO string to prevent Daylight Saving Time conflicts
        ts_utc = ts.astimezone(timezone.utc).isoformat()               
        for col, val in row.items():
            source_raw = col[0] if isinstance(col, tuple) else col
            source_type = SOURCE_MAP.get(source_raw)

            if source_type is None:
                log_note(conn, country, 'unmapped',
                         f"label '{source_raw}' not in SOURCE_MAP at {ts_utc}")
                continue

            rows.append({
                "country":       country,
                "timestamp_utc": ts_utc,
                "source_type":   source_type,
                "mw":            None if pd.isna(val) else float(val),
            })

    with conn.cursor() as cur:
        cur.executemany(
            """
            INSERT INTO generation (country, timestamp_utc, source_type, mw)
            VALUES (%(country)s, %(timestamp_utc)s, %(source_type)s, %(mw)s)
            ON CONFLICT (country, timestamp_utc, source_type) DO UPDATE
                SET mw = EXCLUDED.mw
            """,
            rows,
        )
    conn.commit()
    return len(rows)


def ingest_country_year(client, conn, country: str, year: int):
    start = pd.Timestamp(f"{year}-01-01", tz="Europe/Brussels")
    end   = pd.Timestamp(f"{year+1}-01-01", tz="Europe/Brussels")
    total = 0

    for month_start in pd.date_range(start, end, freq="MS"):
        month_end = month_start + pd.offsets.MonthEnd(1) + pd.Timedelta(days=1)
        month_end = min(month_end, end)

        label = month_start.strftime("%Y-%m")
        print(f"    {country} {label}...", end=" ", flush=True)

        try:
            df = client.query_generation(COUNTRIES[country], start=month_start, end=month_end)
            n = insert_generation(conn, country, df)
            total += n
            print(f"{n} rows")

        except Exception as e:
            print(f"ERROR — {type(e).__name__}: {e}")
            log_note(conn, country, "gap", f"{label}: {str(e)[:200]}")

        time.sleep(2)

    return total

In [4]:
# ── Main ──────────────────────────────────────────────────────────

client = EntsoePandasClient(api_key=ENTSOE_TOKEN)

for country in ["DE_LU", "FR", "ES", "IT", "GB"]:
    for year in [2024, 2025]:
        print(f"\nIngesting {country} {year}...")
        total = ingest_country_year(client, conn, country, year)
        print(f"→ {total} total rows for {country} {year}")

conn.close()



Ingesting DE_LU 2024...
    DE_LU 2024-01... 14136 rows
    DE_LU 2024-02... 13224 rows
    DE_LU 2024-03... 14136 rows
    DE_LU 2024-04... 13680 rows
    DE_LU 2024-05... 14136 rows
    DE_LU 2024-06... 13680 rows
    DE_LU 2024-07... 14136 rows
    DE_LU 2024-08... 14136 rows
    DE_LU 2024-09... 13680 rows
    DE_LU 2024-10... 14155 rows
    DE_LU 2024-11... 13680 rows
    DE_LU 2024-12... 14136 rows
    DE_LU 2025-01... ERROR — NoMatchingDataError: 
→ 166915 total rows for DE_LU 2024

Ingesting DE_LU 2025...
    DE_LU 2025-01... 14136 rows
    DE_LU 2025-02... 12768 rows
    DE_LU 2025-03... 12631 rows
    DE_LU 2025-04... 13680 rows
    DE_LU 2025-05... 14136 rows
    DE_LU 2025-06... 13680 rows
    DE_LU 2025-07... 12648 rows
    DE_LU 2025-08... 12648 rows
    DE_LU 2025-09... 12240 rows
    DE_LU 2025-10... 12665 rows
    DE_LU 2025-11... 12240 rows
    DE_LU 2025-12... 12648 rows
    DE_LU 2026-01... ERROR — NoMatchingDataError: 
→ 156120 total rows for DE_LU 2025

Ingesting

In [7]:
for country in ["DE_LU", "FR", "ES", "IT", "GB"]:
    for year in [2026]:
        print(f"\nIngesting {country} {year}...")
        total = ingest_country_year(client, conn, country, year)
        print(f"→ {total} total rows for {country} {year}")

conn.close()


Ingesting DE_LU 2026...
    DE_LU 2026-01... 12648 rows
    DE_LU 2026-02... 11424 rows
    DE_LU 2026-03... 12631 rows
    DE_LU 2026-04... 12240 rows
    DE_LU 2026-05... 12648 rows
    DE_LU 2026-06... 4012 rows
    DE_LU 2026-07... ERROR — NoMatchingDataError: 
    DE_LU 2026-08... ERROR — NoMatchingDataError: 
    DE_LU 2026-09... ERROR — NoMatchingDataError: 
    DE_LU 2026-10... ERROR — NoMatchingDataError: 
    DE_LU 2026-11... ERROR — NoMatchingDataError: 
    DE_LU 2026-12... ERROR — NoMatchingDataError: 
    DE_LU 2027-01... ERROR — NoMatchingDataError: 
→ 65603 total rows for DE_LU 2026

Ingesting FR 2026...
    FR 2026-01... 12648 rows
    FR 2026-02... 11424 rows
    FR 2026-03... 11888 rows
    FR 2026-04... 11520 rows
    FR 2026-05... 11904 rows
    FR 2026-06... 3525 rows
    FR 2026-07... ERROR — NoMatchingDataError: 
    FR 2026-08... ERROR — NoMatchingDataError: 
    FR 2026-09... ERROR — NoMatchingDataError: 
    FR 2026-10... ERROR — NoMatchingDataError: 
    FR